# Sanday Colab Experiment

Run this notebook with a Google Colab GPU kernel from VS Code. The cells install dependencies, verify the environment, train, evaluate, and display the saved metrics.

In [1]:
!git clone https://github.com/karl4th/sanday.git
%cd sanday

Cloning into 'sanday'...
remote: Enumerating objects: 29, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 29 (delta 2), reused 29 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (29/29), 16.03 KiB | 16.03 MiB/s, done.
Resolving deltas: 100% (2/2), done.
/content/sanday


In [5]:
!git pull origin main

remote: Enumerating objects: 26, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 15 (delta 5), reused 15 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (15/15), 8.81 KiB | 2.20 MiB/s, done.
From https://github.com/karl4th/sanday
 * branch            main       -> FETCH_HEAD
   d7ad506..8d7c41d  main       -> origin/main
Updating d7ad506..8d7c41d
Fast-forward
 .gitignore                              |   4 +-
 README.md                               |  34 +++-
 notebooks/sanday_colab_experiment.ipynb | 332 +++++++++++++++++++++++++++++++-
 results/.gitkeep                        |   1 +
 sanday/__init__.py                      |   2 +-
 sanday/model.py                         | 164 ++++++++++++++++
 sanday/model_v2.py                      | 150 ---------------
 sanday/reporting.py                     |  85 ++++++++
 scripts/evaluate.py                     |  72 ++++---
 scripts/train.py                        |  

In [2]:
from pathlib import Path
import os

def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'configs').is_dir() and (candidate / 'scripts').is_dir() and (candidate / 'sanday').is_dir():
            return candidate
    raise RuntimeError('Could not find Sanday repo root. Open this notebook from inside the sanday repository.')

REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
print('Repo root:', REPO_ROOT)

Repo root: /content/sanday


## Configuration

In [3]:
# Main knobs for this run.
CONFIG = 'configs/sanday_cfc_2m.yaml'  # or: configs/sanday_cfc_2m.yaml
RUN_NAME = 'cfc_2m_seed42'
RUN_DIR = f'results/{RUN_NAME}'
EVAL_DIR = f'{RUN_DIR}_eval'

# If training is already done, set TRAIN = False and keep CHECKPOINT pointing to the best checkpoint.
TRAIN = True
EVALUATE = True
CHECKPOINT = f'{RUN_DIR}/checkpoints/sanday_best.pt'

print('Config:', CONFIG)
print('Run dir:', RUN_DIR)
print('Checkpoint:', CHECKPOINT)

Config: configs/sanday_cfc_2m.yaml
Run dir: results/cfc_2m_seed42
Checkpoint: results/cfc_2m_seed42/checkpoints/sanday_best.pt


## Install Dependencies

In [4]:
%pip install -q -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 63.6 MB/s eta 0:00:0000:01


## Environment Check

In [6]:
import json
import subprocess
import torch

print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print('CUDA version:', torch.version.cuda)
print('Torch version:', torch.__version__)
print()
try:
    print(subprocess.check_output(['nvidia-smi'], text=True))
except Exception as exc:
    print('nvidia-smi failed:', exc)

CUDA: True
GPU: Tesla T4
CUDA version: 12.8
Torch version: 2.11.0+cu128

Thu Jun 18 10:29:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        

## Dataset Check

In [7]:
import kagglehub

dataset_path = kagglehub.dataset_download('prateeknarain/common-voice')
print('Dataset path:', dataset_path)

root = Path(dataset_path)
manifests = sorted(str(path.relative_to(root)) for path in root.rglob('*.tsv'))[:20]
print('TSV files found:')
for item in manifests:
    print(' -', item)

100%|██████████| 15.6G/15.6G [03:27<00:00, 80.9MB/s]

Extracting files...


Dataset path: /root/.cache/kagglehub/datasets/prateeknarain/common-voice/versions/1
TSV files found:
 - Common Voice/Common Voice/English/cv-corpus-21.0-delta-2025-03-14-en/14765303714 14765303717/cv-corpus-21.0-delta-2025-03-14/en/clip_durations.tsv
 - Common Voice/Common Voice/English/cv-corpus-21.0-delta-2025-03-14-en/14765303730 14765303730/cv-corpus-21.0-delta-2025-03-14/en/invalidated.tsv
 - Common Voice/Common Voice/English/cv-corpus-21.0-delta-2025-03-14-en/14765303730 14765303730/cv-corpus-21.0-delta-2025-03-14/en/other.tsv
 - Common Voice/Common Voice/English/cv-corpus-21.0-delta-2025-03-14-en/14765303730 14765303730/cv-corpus-21.0-delta-2025-03-14/en/validated.tsv
 - Common Voice/Common Voice/English/cv-corpus-21.0-delta-2025-03-14-en/14765303731 14765303731/cv-corpus-21.0-delta-2025-03-14/en/reported.tsv
 - Common Voice/Common Voice/English/cv-corpus-21.0-delta-2025-03-14-en/14765303731 14765304003/cv-corpus-21.0-delta-2025-03-14/en/validated_sentences.tsv
 - Common Voice/C

## Train

In [8]:
import shlex
import subprocess
import sys

if TRAIN:
    cmd = [sys.executable, 'scripts/train.py', '--config', CONFIG, '--run-dir', RUN_DIR]
    print('Running:', ' '.join(shlex.quote(part) for part in cmd))
    subprocess.run(cmd, check=True)
else:
    print('TRAIN is False; skipping training.')

Running: /usr/bin/python3 scripts/train.py --config configs/sanday_cfc_2m.yaml --run-dir results/cfc_2m_seed42


CalledProcessError: Command '['/usr/bin/python3', 'scripts/train.py', '--config', 'configs/sanday_cfc_2m.yaml', '--run-dir', 'results/cfc_2m_seed42']' returned non-zero exit status 1.

## Training Results

In [ ]:
import json
import pandas as pd
from IPython.display import display

summary_path = Path(RUN_DIR) / 'summary.json'
metrics_path = Path(RUN_DIR) / 'metrics.csv'

if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print(json.dumps(summary, indent=2))
else:
    print('No summary found:', summary_path)

if metrics_path.exists():
    metrics = pd.read_csv(metrics_path)
    display(metrics.tail(10))
    display(metrics[['epoch', 'train_loss', 'valid_wer', 'valid_cer']].plot(x='epoch', grid=True, figsize=(10, 4)))
else:
    print('No metrics found:', metrics_path)

## Evaluate Best Checkpoint

In [ ]:
if EVALUATE:
    checkpoint_path = Path(CHECKPOINT)
    if not checkpoint_path.exists():
        raise FileNotFoundError(f'Checkpoint not found: {checkpoint_path}')
    cmd = [sys.executable, 'scripts/evaluate.py', '--config', CONFIG, '--checkpoint', str(checkpoint_path), '--run-dir', EVAL_DIR]
    print('Running:', ' '.join(shlex.quote(part) for part in cmd))
    subprocess.run(cmd, check=True)
else:
    print('EVALUATE is False; skipping evaluation.')

## Evaluation Results

In [ ]:
evaluation_path = Path(EVAL_DIR) / 'evaluation.json'
predictions_path = Path(EVAL_DIR) / 'predictions.csv'

if evaluation_path.exists():
    evaluation = json.loads(evaluation_path.read_text())
    print(json.dumps(evaluation, indent=2))
else:
    print('No evaluation found:', evaluation_path)

if predictions_path.exists():
    predictions = pd.read_csv(predictions_path)
    display(predictions.head(20))
else:
    print('No predictions found:', predictions_path)

## Files To Commit Or Share

In [ ]:
important_files = [
    Path(RUN_DIR) / 'config.yaml',
    Path(RUN_DIR) / 'environment.json',
    Path(RUN_DIR) / 'metrics.csv',
    Path(RUN_DIR) / 'metrics.jsonl',
    Path(RUN_DIR) / 'summary.json',
    Path(EVAL_DIR) / 'environment.json',
    Path(EVAL_DIR) / 'evaluation.json',
    Path(EVAL_DIR) / 'predictions.csv',
]

for path in important_files:
    print(('OK   ' if path.exists() else 'MISS '), path)

print('\nCheckpoint is intentionally large and ignored by git:')
print(Path(CHECKPOINT))